# 🔗 Combinación de Datos: Merge, Join y Concat

**Módulo 2** | **Sesión 3** | **Duración estimada: 1h**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-02-data-wrangling/notebooks/02_02_combinacion_datos.ipynb)

---

En esta sesión aprenderemos a **combinar datos provenientes de múltiples fuentes** en un solo DataFrame. En el mundo real, los datos rara vez vienen en un único archivo: necesitamos unir tablas de matrícula con rendimiento, presupuesto con ejecución, encuestas con datos institucionales.

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Nivel | Indicador de logro |
|---|-----------|-------|--------------------||
| 1 | Comprender por qué se necesita combinar datos | Conceptual | Explica escenarios donde se requiere unir datos de múltiples fuentes |
| 2 | Usar pd.concat() para apilar DataFrames | Procedimental | Combina DataFrames vertical y horizontalmente |
| 3 | Usar pd.merge() con distintos tipos de JOIN | Procedimental | Aplica inner, left, right y outer joins correctamente |
| 4 | Realizar merges con múltiples claves | Procedimental | Combina DataFrames usando dos o más columnas como clave |
| 5 | Diagnosticar y depurar problemas en merges | Aplicado | Usa indicator=True y resuelve problemas de claves no coincidentes |

## 💡 ¿Por qué es importante para tu práctica docente?

Como docente de FACES, frecuentemente necesitas cruzar información de diferentes fuentes:

- 🏫 **Gestión académica:** Combinar datos de matrícula con calificaciones para obtener un perfil completo del estudiante
- 📊 **Reportes institucionales:** Unir presupuesto asignado con presupuesto ejecutado para calcular eficiencia
- 🔍 **Investigación:** Cruzar datos de encuestas con datos demográficos de los participantes
- 📈 **Análisis comparativo:** Juntar datos de múltiples períodos académicos para analizar tendencias

> 📣 **En el mundo empresarial y académico, la capacidad de combinar datos de diferentes fuentes es una de las habilidades más valoradas.** Es el equivalente en Python a los `BUSCARV` y tablas dinámicas con múltiples hojas en Excel, pero mucho más poderoso y reproducible.

---

## ⚙️ Configuración inicial

In [ ]:
# ============================================================
# Configuración inicial: bibliotecas
# ============================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('✅ Bibliotecas cargadas exitosamente')

---

## 🧩 Sección 1: ¿Por qué combinar datos?

En la vida real, los datos vienen de **múltiples fuentes**:

| Fuente | Contiene | Ejemplo en FACES |
|--------|----------|------------------|
| Sistema de control de estudios | Matrícula, períodos, estatus | `matricula_faces.csv` |
| Sistema de calificaciones | Notas, asistencia, materias | `rendimiento_academico.csv` |
| Administración | Presupuesto, gastos | `presupuesto_facultad.csv` |
| Encuestas | Satisfacción, percepciones | `encuesta_docente.csv` |

Para responder preguntas como *"¿Cuál es el promedio académico por tipo de ingreso?"*, necesitamos datos de **dos** tablas: matrícula (tipo de ingreso) y rendimiento (promedio).

### Dos operaciones fundamentales

| Operación | Analogía | Cuándo usar |
|-----------|---------|------------|
| **`pd.concat()`** | Apilar hojas de Excel una debajo de otra | Mismas columnas, diferentes filas |
| **`pd.merge()`** | BUSCARV de Excel / JOIN de SQL | Diferentes columnas, clave en común |

---

## 📦 Sección 2: pd.concat() — Apilar DataFrames

`pd.concat()` une DataFrames **apilándolos**, ya sea verticalmente (uno debajo del otro) u horizontalmente (uno al lado del otro).

- **`axis=0`** (por defecto): Apila filas (vertical). Útil para combinar datos de diferentes períodos
- **`axis=1`**: Apila columnas (horizontal). Útil para añadir columnas calculadas

In [ ]:
# ============================================================
# Ejemplo 1: Concat vertical (axis=0)
# ============================================================
# Imaginemos que tenemos datos de estudiantes divididos en dos archivos

# Creamos dos DataFrames de ejemplo
grupo_a = pd.DataFrame({
    'estudiante': ['Ana', 'Carlos', 'María'],
    'carrera': ['Economia', 'Contaduria', 'Administracion'],
    'promedio': [15.5, 12.8, 17.2]
})

grupo_b = pd.DataFrame({
    'estudiante': ['Pedro', 'Lucía', 'José'],
    'carrera': ['Economia', 'Relaciones Industriales', 'Contaduria'],
    'promedio': [14.1, 16.0, 11.5]
})

print('Grupo A:')
print(grupo_a)
print('\nGrupo B:')
print(grupo_b)

# Concat vertical: apilar uno debajo del otro
todos = pd.concat([grupo_a, grupo_b], ignore_index=True)

print('\n🔗 Resultado de pd.concat([grupo_a, grupo_b]):')
print(todos)
print(f'\n📌 De {len(grupo_a)} + {len(grupo_b)} filas pasamos a {len(todos)} filas')

In [ ]:
# ============================================================
# Ejemplo 2: Concat horizontal (axis=1)
# ============================================================
# Añadir columnas calculadas a un DataFrame

datos_basicos = pd.DataFrame({
    'estudiante': ['Ana', 'Carlos', 'María'],
    'promedio': [15.5, 12.8, 17.2]
})

datos_extra = pd.DataFrame({
    'asistencia': [90, 75, 95],
    'aprobadas': [6, 4, 7]
})

# Concat horizontal: columnas una al lado de otra
completo = pd.concat([datos_basicos, datos_extra], axis=1)

print('Datos básicos + datos extra (axis=1):')
print(completo)
print('\n📌 axis=1 añade columnas. Requiere que ambos DataFrames tengan el mismo índice.')

In [ ]:
# ============================================================
# Ejemplo práctico: Concat con datos reales
# ============================================================
# Dividimos el dataset de rendimiento por carrera y lo volvemos a unir

df_rendimiento = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

# Dividir por carrera
df_economia = df_rendimiento[df_rendimiento['carrera'] == 'Economia']
df_contaduria = df_rendimiento[df_rendimiento['carrera'] == 'Contaduria']
df_admin = df_rendimiento[df_rendimiento['carrera'] == 'Administracion']
df_rrii = df_rendimiento[df_rendimiento['carrera'] == 'Relaciones Industriales']

print(f'Economía: {len(df_economia)} filas')
print(f'Contaduría: {len(df_contaduria)} filas')
print(f'Administración: {len(df_admin)} filas')
print(f'Relaciones Industriales: {len(df_rrii)} filas')

# Reconstituir con concat
df_reconstituido = pd.concat([df_economia, df_contaduria, df_admin, df_rrii], 
                              ignore_index=True)

print(f'\n🔗 Reconstituido: {len(df_reconstituido)} filas')
print(f'✅ ¿Igual al original? {len(df_reconstituido) == len(df_rendimiento)}')

---

## 🔀 Sección 3: pd.merge() — Joins tipo SQL

`pd.merge()` combina dos DataFrames usando una **columna en común** (clave). Funciona igual que los JOINs de SQL.

### Tipos de merge

| Tipo | SQL equivalente | Resultado | Diagrama |
|------|----------------|-----------|----------|
| `inner` | INNER JOIN | Solo filas con coincidencia en **ambas** tablas | A ∩ B |
| `left` | LEFT JOIN | **Todas** las filas de la izquierda + coincidencias de la derecha | A + (A ∩ B) |
| `right` | RIGHT JOIN | **Todas** las filas de la derecha + coincidencias de la izquierda | (A ∩ B) + B |
| `outer` | FULL OUTER JOIN | **Todas** las filas de ambas tablas | A ∪ B |

### Analogía con Excel

`pd.merge()` es como un **BUSCARV** (VLOOKUP) potenciado:
- En Excel: `=BUSCARV(A2, HojaCalificaciones!A:C, 3, FALSO)`
- En pandas: `pd.merge(matricula, calificaciones, on='estudiante_id')`

In [ ]:
# ============================================================
# Ejemplo 3: Merge básico con datos de ejemplo
# ============================================================

# Tabla 1: Estudiantes y sus carreras
estudiantes = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'nombre': ['Ana', 'Carlos', 'María', 'Pedro', 'Lucía'],
    'carrera': ['Economia', 'Contaduria', 'Administracion', 'Economia', 'Contaduria']
})

# Tabla 2: Calificaciones (nota: el estudiante 5 no tiene calificación,
# y hay un estudiante 6 que no está en la tabla de estudiantes)
calificaciones = pd.DataFrame({
    'id': [1, 2, 3, 4, 6],
    'promedio': [15.5, 12.8, 17.2, 14.1, 16.0],
    'asistencia': [90, 75, 95, 80, 88]
})

print('Tabla de estudiantes:')
print(estudiantes)
print('\nTabla de calificaciones:')
print(calificaciones)

In [ ]:
# ============================================================
# Los 4 tipos de merge
# ============================================================

# INNER: solo los que están en AMBAS tablas
inner = pd.merge(estudiantes, calificaciones, on='id', how='inner')
print('1. INNER JOIN (solo coincidencias):')
print(inner)
print(f'   Filas: {len(inner)} (Lucía e id=6 se pierden)\n')

# LEFT: todos los de la tabla izquierda
left = pd.merge(estudiantes, calificaciones, on='id', how='left')
print('2. LEFT JOIN (todos los estudiantes):')
print(left)
print(f'   Filas: {len(left)} (Lucía aparece con NaN en calificaciones)\n')

# RIGHT: todos los de la tabla derecha
right = pd.merge(estudiantes, calificaciones, on='id', how='right')
print('3. RIGHT JOIN (todas las calificaciones):')
print(right)
print(f'   Filas: {len(right)} (id=6 aparece con NaN en nombre)\n')

# OUTER: todos los registros de ambas tablas
outer = pd.merge(estudiantes, calificaciones, on='id', how='outer')
print('4. OUTER JOIN (todos los registros):')
print(outer)
print(f'   Filas: {len(outer)} (nadie se pierde, NaN donde no hay coincidencia)')

---

## 🏫 Sección 4: Caso práctico — Datos universitarios

Ahora trabajaremos con los datasets reales de FACES. El objetivo es combinar información de **matrícula** con **rendimiento académico** para obtener una visión integral por carrera.

In [ ]:
# ============================================================
# Cargar ambos datasets
# ============================================================

df_matricula = pd.read_csv('../datasets/universidad/matricula_faces.csv')
df_rendimiento = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

print('=== Matrícula FACES ===')
print(f'Dimensiones: {df_matricula.shape}')
print(f'Columnas: {list(df_matricula.columns)}')
print(f'Carreras: {df_matricula["carrera"].unique().tolist()}')

print('\n=== Rendimiento Académico ===')
print(f'Dimensiones: {df_rendimiento.shape}')
print(f'Columnas: {list(df_rendimiento.columns)}')
print(f'Carreras: {df_rendimiento["carrera"].unique().tolist()}')

print('\n🔗 Columna en común: "carrera"')

In [ ]:
# ============================================================
# Crear tablas resumen de cada dataset
# ============================================================

# Resumen de matrícula: conteo por carrera y estatus
resumen_matricula = df_matricula.groupby('carrera').agg(
    total_registros=('carrera', 'count'),
    periodos=('periodo', 'nunique'),
    pct_femenino=('genero', lambda x: (x == 'F').mean() * 100)
).round(1)

# Tasa de retiro por carrera
retiro = df_matricula.groupby('carrera')['estatus'].apply(
    lambda x: (x == 'Retirado').mean() * 100
).round(1).rename('pct_retirados')

resumen_matricula = resumen_matricula.join(retiro)

print('📊 Resumen de matrícula por carrera:')
print(resumen_matricula)

In [ ]:
# ============================================================
# Resumen de rendimiento por carrera
# ============================================================

resumen_rendimiento = df_rendimiento.groupby('carrera').agg(
    total_estudiantes=('estudiante_id', 'count'),
    promedio_general=('promedio', 'mean'),
    asistencia_promedio=('asistencia_pct', 'mean'),
    pct_becados=('beca', lambda x: (x == True).mean() * 100)
).round(2)

print('📊 Resumen de rendimiento por carrera:')
print(resumen_rendimiento)

In [ ]:
# ============================================================
# Merge: combinar ambas tablas resumen
# ============================================================

# Ambas tablas tienen 'carrera' como índice, así que 
# podemos hacer merge por índice o resetearlo primero

ficha_carrera = resumen_matricula.join(resumen_rendimiento)

print('🔗 Ficha completa por carrera (matrícula + rendimiento):')
print('=' * 80)
print(ficha_carrera)

print('\n📌 Ahora tenemos una vista unificada que combina:')
print('   - De matrícula: total registros, periodos, % femenino, % retiros')
print('   - De rendimiento: estudiantes, promedio, asistencia, % becados')

In [ ]:
# ============================================================
# Alternativa: merge usando pd.merge() explícito
# ============================================================
# Si las tablas resumen no comparten índice, usamos merge

resumen_mat = df_matricula.groupby('carrera').size().reset_index(name='registros_matricula')
resumen_rend = df_rendimiento.groupby('carrera').agg(
    promedio=('promedio', 'mean'),
    estudiantes=('estudiante_id', 'count')
).reset_index().round(2)

print('Tabla 1 (matrícula):')
print(resumen_mat)
print('\nTabla 2 (rendimiento):')
print(resumen_rend)

# Merge explícito con pd.merge
combinado = pd.merge(resumen_mat, resumen_rend, on='carrera', how='inner')

print('\n🔗 Resultado del merge:')
print(combinado)

---

## 🔑 Sección 5: Merge con múltiples claves

A veces necesitamos combinar datos usando **más de una columna** como clave. Por ejemplo, si queremos combinar datos agrupados por carrera **y** género.

In [ ]:
# ============================================================
# Merge con múltiples claves: carrera + genero
# ============================================================

# Resumen de matrícula por carrera y género
mat_carrera_genero = df_matricula.groupby(['carrera', 'genero']).agg(
    registros=('carrera', 'count'),
    pct_retirados=('estatus', lambda x: (x == 'Retirado').mean() * 100)
).reset_index().round(1)

# Resumen de rendimiento por carrera y género
rend_carrera_genero = df_rendimiento.groupby(['carrera', 'genero']).agg(
    promedio=('promedio', 'mean'),
    asistencia=('asistencia_pct', 'mean')
).reset_index().round(2)

print('Matrícula por carrera + género:')
print(mat_carrera_genero.head())
print('\nRendimiento por carrera + género:')
print(rend_carrera_genero.head())

# Merge con DOS claves
combinado_doble = pd.merge(
    mat_carrera_genero, 
    rend_carrera_genero, 
    on=['carrera', 'genero'],  # <-- múltiples claves
    how='inner'
)

print('\n🔗 Merge con doble clave (carrera + género):')
print(combinado_doble)

print('\n📌 Usar múltiples claves evita combinaciones incorrectas')
print('   (ej: unir mujeres de Economía con hombres de Contaduría)')

---

## 🔧 Sección 6: Diagnóstico de merges

Los merges pueden fallar silenciosamente: no dan error, pero el resultado no es el esperado. Veamos cómo diagnosticar problemas comunes.

In [ ]:
# ============================================================
# indicator=True para diagnosticar el merge
# ============================================================

# Creamos tablas donde no todas las claves coinciden
tabla_a = pd.DataFrame({
    'carrera': ['Economia', 'Contaduria', 'Administracion', 'Relaciones Industriales'],
    'promedio': [13.5, 12.8, 14.2, 13.0]
})

tabla_b = pd.DataFrame({
    'carrera': ['Economia', 'Contaduria', 'Derecho', 'Ingenieria'],
    'estudiantes': [800, 900, 500, 600]
})

# Merge con indicator=True
resultado = pd.merge(tabla_a, tabla_b, on='carrera', how='outer', indicator=True)

print('🔍 Merge con indicator=True:')
print(resultado)
print()
print('📊 Resumen del diagnóstico:')
print(resultado['_merge'].value_counts())
print()
print('Interpretación:')
print('  - both:       Registro presente en ambas tablas')
print('  - left_only:  Solo en la tabla izquierda (no tiene coincidencia en la derecha)')
print('  - right_only: Solo en la tabla derecha (no tiene coincidencia en la izquierda)')

In [ ]:
# ============================================================
# Problema común: duplicados que multiplican filas
# ============================================================

# Si una clave aparece múltiples veces en ambas tablas,
# el merge genera un producto cartesiano (muchas filas)

tabla_dup_a = pd.DataFrame({
    'carrera': ['Economia', 'Economia', 'Contaduria'],
    'estudiante': ['Ana', 'Carlos', 'María']
})

tabla_dup_b = pd.DataFrame({
    'carrera': ['Economia', 'Economia', 'Contaduria'],
    'materia': ['Micro', 'Macro', 'Auditoría']
})

resultado_dup = pd.merge(tabla_dup_a, tabla_dup_b, on='carrera')

print('⚠️ Cuidado con duplicados en la clave:')
print(f'\nTabla A: {len(tabla_dup_a)} filas')
print(tabla_dup_a)
print(f'\nTabla B: {len(tabla_dup_b)} filas')
print(tabla_dup_b)
print(f'\nResultado del merge: {len(resultado_dup)} filas (!)')
print(resultado_dup)
print('\n📌 2 economías x 2 economías = 4 combinaciones. Se multiplicaron las filas.')
print('   Solución: agrupa/resume antes de hacer merge, o verifica unicidad de la clave.')

In [ ]:
# ============================================================
# validate: verificar unicidad antes del merge
# ============================================================

# pd.merge tiene el parámetro validate para verificar la relación
# 'one_to_one': ambas claves son únicas
# 'one_to_many': clave izquierda única, derecha puede repetirse
# 'many_to_one': clave izquierda puede repetirse, derecha única

# Ejemplo correcto: merge de resúmenes (una fila por carrera)
try:
    resultado_validado = pd.merge(
        resumen_mat, resumen_rend, on='carrera', 
        how='inner', validate='one_to_one'
    )
    print('✅ Merge validado como one_to_one: correcto')
    print(resultado_validado)
except Exception as e:
    print(f'❌ Error de validación: {e}')

---

## ✏️ Ejercicios

Ahora es tu turno de practicar la combinación de datos.

### Ejercicio 1: Merge de presupuesto con metas

1. Carga `presupuesto_facultad.csv` y crea un resumen con el promedio de `ejecutado` por `partida`
2. Crea manualmente un DataFrame con metas por partida:
   ```python
   metas = pd.DataFrame({
       'partida': ['Personal', 'Materiales', 'Equipos', 'Mantenimiento', 'Servicios'],
       'meta_ejecutado': [800000, 150000, 200000, 120000, 100000]
   })
   ```
3. Haz un merge para comparar el promedio real vs la meta
4. Calcula una columna de **cumplimiento** (ejecutado / meta * 100)

In [ ]:
# ============================================================
# Ejercicio 1: Merge de presupuesto con metas
# ============================================================

# 1. Cargar presupuesto y calcular promedio por partida
# df_presupuesto = pd.read_csv('../datasets/universidad/presupuesto_facultad.csv')
# resumen_pres = ...

# 2. Crear DataFrame de metas
# metas = pd.DataFrame({...})

# 3. Merge
# Tu código aquí

# 4. Calcular cumplimiento
# Tu código aquí

### Ejercicio 2: Concat + merge con encuesta docente

1. Carga `encuesta_docente.csv`
2. Divide el DataFrame en dos partes: docentes del departamento de `Economia` y docentes de `Administracion`
3. Usa `pd.concat()` para volver a unirlos
4. Crea un DataFrame resumen de satisfacción por departamento y haz merge con este DataFrame de información adicional:
   ```python
   info_depto = pd.DataFrame({
       'departamento': ['Economia', 'Administracion', 'Contaduria', 
                        'Relaciones Industriales', 'Estadistica', 'Matematica'],
       'escuela': ['Economia', 'Administracion', 'Contaduria', 
                   'Relaciones Industriales', 'Básico', 'Básico'],
       'num_aulas': [12, 15, 10, 8, 6, 5]
   })
   ```

In [ ]:
# ============================================================
# Ejercicio 2: Concat + merge con encuesta docente
# ============================================================

# 1. Cargar encuesta
# df_encuesta = pd.read_csv('../datasets/universidad/encuesta_docente.csv')

# 2. Dividir por departamento
# Tu código aquí

# 3. Concat
# Tu código aquí

# 4. Resumen de satisfacción + merge con info_depto
# Tu código aquí

### Ejercicio 3: Ficha integral por carrera

Crea una "ficha por carrera" combinando métricas de `matricula_faces.csv` y `rendimiento_academico.csv`:

Para cada carrera, incluye:
- De matrícula: total registros, % retirados, % femenino, periodos disponibles
- De rendimiento: total estudiantes, promedio, asistencia promedio, % con beca, % que trabajan

El resultado debe ser un único DataFrame con una fila por carrera y todas estas columnas.

In [ ]:
# ============================================================
# Ejercicio 3: Ficha integral por carrera
# ============================================================

# 1. Resumen de matrícula por carrera
# Tu código aquí

# 2. Resumen de rendimiento por carrera
# Tu código aquí

# 3. Merge para crear la ficha completa
# Tu código aquí

---

## 📋 Resumen

En esta sesión aprendimos:

| Operación | Función | Uso principal |
|-----------|--------|---------------|
| Apilar vertical | `pd.concat([df1, df2])` | Unir datos con las mismas columnas |
| Apilar horizontal | `pd.concat([df1, df2], axis=1)` | Añadir columnas de otro DataFrame |
| Inner join | `pd.merge(df1, df2, on='clave', how='inner')` | Solo coincidencias en ambas tablas |
| Left join | `pd.merge(df1, df2, on='clave', how='left')` | Todos los de la izquierda |
| Right join | `pd.merge(df1, df2, on='clave', how='right')` | Todos los de la derecha |
| Outer join | `pd.merge(df1, df2, on='clave', how='outer')` | Todos los registros |
| Múltiples claves | `pd.merge(df1, df2, on=['col1', 'col2'])` | Combinar por dos o más columnas |
| Diagnóstico | `pd.merge(..., indicator=True)` | Verificar coincidencias |
| Validación | `pd.merge(..., validate='one_to_one')` | Prevenir productos cartesianos |

### Consejos prácticos

1. **Siempre verifica las dimensiones** antes y después del merge
2. **Usa `indicator=True`** en merges complejos para diagnosticar problemas
3. **Agrega antes de hacer merge** para evitar explosiones de filas
4. **Verifica que las claves coincidan** (mayúsculas, espacios, acentos)
5. **Prefiere `left` join** cuando quieres mantener todos los registros de tu tabla principal

## 📚 Referencias

### Recursos recomendados

1. **pandas Documentation — Merge, join, concatenate:** [https://pandas.pydata.org/docs/user_guide/merging.html](https://pandas.pydata.org/docs/user_guide/merging.html)
2. **Real Python — Combining Data in pandas:** [https://realpython.com/pandas-merge-join-and-concat/](https://realpython.com/pandas-merge-join-and-concat/)
3. **Kaggle Learn — Data Cleaning (Joining Data):** [https://www.kaggle.com/learn/data-cleaning](https://www.kaggle.com/learn/data-cleaning)
4. **Visual Explanation of SQL Joins:** [https://joins.spathon.com/](https://joins.spathon.com/)

### Bibliografía

- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media. Capítulo 8: Data Wrangling.
- VanderPlas, J. (2016). *Python Data Science Handbook.* O'Reilly Media.

---

*Notebook desarrollado para el programa de Formación Docente en Ciencia de Datos y Business Intelligence — FACES, Universidad Catolica Andres Bello (UCAB).*